<img src="https://fedesarrollo.org.co/logo-Fede.png" alt="Fedesarrollo" style="float: right; width: 300px; height: auto;">
<img src="https://upload.wikimedia.org/wikipedia/commons/f/f9/Logo_BID_Espa%C3%B1ol.png" alt="Banco Interamericano de Desarrollo" style="float: right; width: 220px; height: auto;">

# Índice normalizado del crecimiento exportador a veinte años

Editor: Juan Diego Heredia Niño - jheredia@fedesarrollo.org.co

Septiembre 2026

### Importar librerías

La siguiente celda marca el inicio de un nuevo bloque de trabajo independiente:

**Librerías utilizadas:**
- `pandas` y `numpy`: para manipulación y análisis de datos estructurados.

In [1]:
import pandas as pd
import numpy as np

### Configuración de rutas

Carga las rutas del proyecto y las opciones globales desde `config.py`, instalado como
paquete editable en la raíz del proyecto.

In [2]:
from config import raw, temp

### Parámetros

El insumo es un extracto de WITS con las exportaciones totales anuales de los cuatro países de
referencia hacia el mundo. Las ventanas son de veinte períodos, que es el horizonte sobre el que
se proyecta la ganancia exportadora.

In [3]:
archivo_wits = raw / 'comentarios' / '3004985_1C4043D0-F' / 'DataJobID-3004985_3004985_howExportGrows.csv'

largo_ventana = 20

### Lectura del extracto de exportaciones totales

El archivo trae, para cada país y año, el valor exportado hacia el agregado mundial. Se retienen
las filas del producto agregado `Total`, del socio `WLD` y del flujo de exportación, que son las
que contienen la serie de interés.

El archivo viene codificado en `latin-1`, no en `utf-8`.

In [4]:
df_wits = pd.read_csv(archivo_wits, encoding='latin-1')
df_wits['ProductCode'] = df_wits['ProductCode'].astype(str).str.strip()

df_exportaciones = (
    df_wits
    .query("ProductCode == 'Total' and PartnerISO3 == 'WLD' and TradeFlowName == 'Export'")
    [['ReporterName', 'Year', 'TradeValue in 1000 USD']]
    .rename(columns={'ReporterName': 'pais', 'Year': 'anio', 'TradeValue in 1000 USD': 'exportaciones'})
    .sort_values(['pais', 'anio'])
    .reset_index(drop=True)
)

df_cobertura = (
    df_exportaciones
    .groupby('pais')
    .agg(observaciones=('anio', 'size'), anio_inicial=('anio', 'min'), anio_final=('anio', 'max'))
    .assign(ventanas=lambda d: d['observaciones'] - largo_ventana + 1)
)

print(df_cobertura.to_string())
print(f"\nVentanas totales: {df_cobertura['ventanas'].sum()}")

                    observaciones  anio_inicial  anio_final  ventanas
pais                                                                 
Colombia                       34          1991        2024        15
Costa Rica                     31          1994        2024        12
Dominican Republic             26          1996        2024         7
Mexico                         35          1990        2024        16

Ventanas totales: 50


### Ventanas móviles

Para cada país se construyen todas las ventanas de veinte observaciones consecutivas y cada una se
normaliza dividiendo por su primer valor, de modo que el índice arranca en 1 por construcción.

**Las ventanas son posicionales, no de calendario.** La serie de República Dominicana tiene un
hueco entre 1998 y 2000, y el método original lo atraviesa tratando las observaciones como
consecutivas. Respetarlo importa: exigir años continuos deja 48 ventanas en vez de 50 y desvía el
índice hasta 0,066 respecto del archivado.

In [5]:
def ventanas_normalizadas(serie, largo):
    """Devuelve todas las ventanas de `largo` observaciones consecutivas, cada una dividida por su primer valor."""
    valores = np.asarray(serie, dtype=float)
    return [valores[i:i + largo] / valores[i] for i in range(len(valores) - largo + 1)]


ventanas = list()
for pais, df_pais in df_exportaciones.groupby('pais'):
    ventanas.extend(ventanas_normalizadas(df_pais['exportaciones'], largo_ventana))

matriz_ventanas = np.array(ventanas)
print(f"Matriz de ventanas: {matriz_ventanas.shape[0]} ventanas x {matriz_ventanas.shape[1]} periodos")

Matriz de ventanas: 50 ventanas x 20 periodos


### Índice promedio

El índice es el promedio simple de todas las ventanas de todos los países, período a período. Al
promediar se extrae la forma funcional típica del crecimiento exportador y se diluyen los auges y
las crisis propios de cada país y de cada momento.

In [6]:
df_indice = pd.DataFrame({
    'periodo': np.arange(largo_ventana),
    'indice_normalizado': matriz_ventanas.mean(axis=0),
})

print(df_indice.to_string(index=False))

 periodo  indice_normalizado
       0                1.00
       1                1.21
       2                1.35
       3                1.53
       4                1.66
       5                1.79
       6                1.94
       7                2.09
       8                2.24
       9                2.41
      10                2.59
      11                2.75
      12                2.93
      13                3.16
      14                3.32
      15                3.48
      16                3.65
      17                3.82
      18                3.95
      19                4.08


### Comprobación contra el archivo original

El proyecto original entregaba este índice como un libro de Excel, sin el código que lo produjo.
La celda contrasta el índice reconstruido contra ese archivo.

In [7]:
df_indice_original = pd.read_excel(raw / 'modelo' / 'indices_normalizados_20anios.xlsx')

diferencia = np.abs(df_indice['indice_normalizado'].to_numpy() - df_indice_original['indice_normalizado'].to_numpy())

print(f"Diferencia maxima contra el archivo original: {diferencia.max():.3e}")
print(f"Diferencia mediana:                           {np.median(diferencia):.3e}")

Diferencia maxima contra el archivo original: 1.332e-15
Diferencia mediana:                           4.441e-16


### Exportación

El índice queda disponible para la priorización por industria y para las figuras de la
presentación, que hasta ahora leían el libro de Excel archivado.

In [8]:
(temp / 'modelo').mkdir(parents=True, exist_ok=True)
df_indice.to_parquet(temp / 'modelo' / 'indices_normalizados_20anios.parquet', index=False)